# 📗 부분 실패·폴백·이어 하기

교안 02에서는 끝나지 않는 반복과, 잠깐 끊기거나 응답하지 않는 외부 API를 다뤘습니다. 이 교안은 같은 **주간 시장 브리핑 팀**(생활가전 브랜드 **온결**의 마케팅팀, 수업용 가상 회사)으로 팀이 **일부만 실패하거나, 재시도로도 회복되지 않거나, 중간에 멈추거나, 같은 일을 두 번 하는** 경우를 다룹니다. 이번 팀은 조사·종합·작성 노드가 LLM을 부르고, 완성한 브리핑을 사내 게시판에 올립니다. 이 팀이 쓰는 외부 API도 일부러 실패하게 만들어 두었습니다.

<img src="images/00_lesson03_goals.png" width="1050" alt="오늘 배울 것 네 가지: 1 실패를 기록하고 계속·보류·중단 판단(조사원 셋 중 한 명이 실패 쪽지를 상자에 넣고 신호등으로 진행을 정함), 2 막힌 길은 폴백 경로로(차단된 길 대신 초록 우회로로 지난주 자료에 닿음), 3 멈춘 곳에서 이어 하고 한 번만 게시(깃발을 꽂아 둔 정거장을 건너뛰고 고장 난 정거장부터 다시, 게시판에는 한 장만), 4 잘못된 결정에서 새 갈래로(갈림길 표지판을 고쳐 새 길로 다시 가고 원래 길에서 나간 메일은 그대로)">

그림의 번호 1~4가 아래 학습 목표 1~4입니다.

**학습 목표**

1. **병렬 조사 하나가 실패해도 팀이 끝까지 가게 합니다.** 실패를 State에 기록하고, 실패 뒤 축소 진행·발행 보류·중단을 판단합니다.
2. **재시도로도 회복되지 않는 노드는 `error_handler`로 폴백 경로에 보냅니다.** 핸들러가 `Command(goto=...)`로 다음 노드를 정해야 하는 이유를 확인합니다.
3. **체크포인터로 멈춘 곳에서 이어 하고, 게시 같은 부작용은 멱등 키로 한 번만 실행합니다.**
4. **잘못된 결정은 과거 체크포인트에서 값만 고쳐 새 갈래(fork)로 다시 실행합니다.**

## 1. 실습 준비

### 1-1. 라이브러리·모델·Langfuse 설정

`.env`에 `OPENAI_API_KEY`와 `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_BASE_URL`을 설정하고 새 커널에서 시작합니다. Langfuse 키 발급은 교안 01의 2-1, 화면 여는 법은 교안 01의 5-4와 같습니다.

라이브러리·모델 설정, 실행 설정 함수 `make_config`·`trace_url`, 궤적 모으기 함수 `stream_trail`은 교안 02와 같아서 한 셀에 모았습니다. 시연마다 `session_id`와 `thread_id`에 같은 값을 넘겨, 한 시연의 실행들이 체크포인터에서도 Langfuse 세션에서도 함께 묶이게 합니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import os
import json
import operator
from datetime import datetime
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, RetryPolicy
from IPython.display import Image, Markdown

# material_dir는 output이 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 단원 폴더에서 가까운 .env를 찾아 API 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)
print("답변 모델:", model_name)

# Langfuse 클라이언트: LANGFUSE_* 환경 변수로 연결합니다. 키가 틀리면 False가 나옵니다.
langfuse = get_client()
print("Langfuse 인증:", langfuse.auth_check())

# 노트북을 다시 실행해도 세션·thread_id 이름이 겹치지 않게 실행 시각을 붙입니다.
run_stamp = datetime.now().strftime("%m%d-%H%M%S")


# [보조 함수] 실행 설정 만들기: 실행마다 새 Langfuse 핸들러와 세션·thread_id·태그를 담습니다.
def make_config(session_id, thread_id, tags, recursion_limit=30):
    """Langfuse 콜백·세션·thread_id·태그·단계 상한을 담은 그래프 실행 설정을 만듭니다."""
    return {
        # 실행마다 새 핸들러를 넣어, 실행이 끝난 뒤 그 실행의 trace id를 핸들러에서 읽습니다.
        "callbacks": [CallbackHandler()],
        # trace 이름: Langfuse 목록에서 브리핑 팀 실행을 이 이름으로 찾습니다.
        "run_name": "briefing-team",
        # 그래프가 실행할 수 있는 단계 수의 상한입니다. config 최상위 키입니다.
        "recursion_limit": recursion_limit,
        # thread_id는 체크포인터가 State를 저장하는 열쇠입니다. 실패 뒤 멈춘 단계부터 다시 실행할 때 이 값으로 State를 찾습니다.
        "configurable": {"thread_id": thread_id},
        # langfuse_로 시작하는 키는 trace의 세션·태그로 들어갑니다.
        "metadata": {"langfuse_session_id": session_id, "langfuse_tags": tags},
    }


# [보조 함수] trace 주소: 보내지 않은 기록을 먼저 보내고 마지막 trace의 주소를 돌려줍니다.
def trace_url(config):
    """실행에 쓴 핸들러의 마지막 trace 주소를 반환합니다."""
    # Langfuse는 기록을 모아서 보내므로 flush 전에는 화면에 trace가 없을 수 있습니다.
    langfuse.flush()
    handler = config["callbacks"][0]
    return langfuse.get_trace_url(trace_id=handler.last_trace_id)


# [보조 함수] 궤적 모으기: 노드가 끝날 때마다 이름을 trail에 넣고 마지막 State를 반환합니다.
def stream_trail(app, inputs, config, trail):
    """그래프를 실행하며 끝난 노드 이름을 trail에 순서대로 넣고 마지막 State를 반환합니다."""
    final = None
    for mode, chunk in app.stream(inputs, config, stream_mode=["updates", "values"]):
        if mode == "updates":
            # updates는 {노드 이름: 그 노드가 바꾼 값} 모양입니다.
            for node_name in chunk:
                trail.append(node_name)
        else:
            # values는 단계마다 State 전체입니다. 마지막 것이 최종 결과입니다.
            final = chunk

    return final

### 1-2. 브리핑 자료와 외부 API 함수

교안 02와 같은 자료입니다. 경쟁사 8곳의 공기청정기 가격, 업계 뉴스 3건, AP-500 고객 리뷰 3건을 노트북 안에 둡니다. 회사·제품·기사는 모두 수업용으로 만든 가상 자료입니다. 자료와 외부 API 함수는 시연용 준비물이라 한 셀에 모았습니다.

**외부 API 함수:** 아래 네 함수는 **수업용으로 일부러 실패하게 만든 함수**입니다. 실제 서비스라면 HTTP 요청이 들어갈 자리입니다.

| 함수 | 하는 일 | 넣어 둔 실패 (실패 스위치) |
|---|---|---|
| `fetch_prices()` | 경쟁사 가격 목록 | 앞의 `price_failures`번은 `ConnectionError("가격 API 연결 끊김")` (일시 오류). `price_format_changed`면 매번 `KeyError` (영구 오류) |
| `fetch_news()` (async) | 업계 뉴스 | 없음. async 함수라 `await`로 부릅니다 |
| `fetch_reviews()` | 고객 리뷰 | 스위치 없이 매번 `KeyError`. 예전 형식의 키 `"reviews"`를 읽기 때문입니다 (영구 오류) |
| `post_briefing(text, key)` | 사내 게시판에 글 올리기 (부작용) | `board_down`이면 연결 실패, `drop_response`면 글은 올라가고 응답만 한 번 끊김 |

`CALLS`는 함수가 불린 횟수, `POSTED`는 게시판에 올라간 글, `FAULTS`는 실패 스위치입니다. 스위치는 모두 꺼진 채로 시작합니다. **각 시연은 `reset_demo()`로 기록을 비운 뒤 그 시연에 필요한 실패 하나만 켜고 시작합니다.** 그래서 셀을 다시 실행해도 결과가 같습니다.

In [ ]:
# 이번 브리핑 주차(ISO 주차)와 우리 제품
WEEK = "2026-41"
OUR_PRODUCT = {"모델": "온결 AP-500", "가격": 429000}

# 경쟁사 8곳의 대표 공기청정기 가격. 금액 단위는 원입니다.
COMPETITOR_PRICES = [
    {"회사": "하늘가전", "모델": "클린맥스 S7", "이번 주": 389000, "지난주": 419000},
    {"회사": "바람전자", "모델": "에어핏 500", "이번 주": 349000, "지난주": 399000},
    {"회사": "맑음테크", "모델": "퓨어온 L", "이번 주": 449000, "지난주": 449000},
    {"회사": "숲결", "모델": "포레스트 360", "이번 주": 412000, "지난주": 412000},
    {"회사": "새벽홈", "모델": "던에어 7", "이번 주": 299000, "지난주": 319000},
    {"회사": "다온전자", "모델": "다온 클린 9", "이번 주": 469000, "지난주": 469000},
    {"회사": "이음가전", "모델": "이음 퓨어 S", "이번 주": 359000, "지난주": 379000},
    {"회사": "누리케어", "모델": "누리 에어 X", "이번 주": 398000, "지난주": 398000},
]

# 업계 뉴스 3건 (수업용 가상 기사)
NEWS = [
    {"날짜": "2026-10-05", "제목": "바람전자, 에어핏 500 가을 할인 10월 19일까지", "요약": "정가 399,000원에서 349,000원으로 50,000원 할인"},
    {"날짜": "2026-10-02", "제목": "하늘가전, 필터 정기 배송 구독 시작", "요약": "월 9,900원에 6개월마다 필터를 교체해 줍니다"},
    {"날짜": "2026-09-30", "제목": "10월 중순부터 미세먼지 나쁨 일수 늘어날 전망", "요약": "공기청정기 수요가 10월 셋째 주부터 늘 것으로 예상"},
]

# 리뷰 API의 새 응답 형식. 예전에는 {"reviews": [...]}였는데 {"data": {"items": [...]}}로 바뀌었습니다.
REVIEW_API_RESPONSE = {"data": {"items": [
    {"별점": 5, "내용": "소음이 작아 침실에 두고 씁니다."},
    {"별점": 4, "내용": "필터 교체 알림이 정확합니다."},
    {"별점": 2, "내용": "앱 연결이 자주 끊깁니다."},
]}}

# 조사 작업 이름과 브리핑에 쓰는 한국어 이름
SOURCE_NAMES = {"prices": "경쟁사 가격", "news": "업계 뉴스", "reviews": "고객 리뷰"}


# 함수별 호출 횟수, 게시판에 올라간 글, 실패 스위치(처음에는 모두 꺼짐)
CALLS = {"fetch_prices": 0, "fetch_news": 0, "fetch_reviews": 0, "post_briefing": 0}
POSTED = []
FAULTS = {"price_failures": 0, "price_format_changed": False, "board_down": False, "drop_response": False}


# [보조 함수] 시연 준비: 호출 수와 게시판을 비우고 실패 스위치를 모두 끕니다.
def reset_demo():
    """CALLS를 0으로, POSTED를 빈 목록으로 되돌리고 FAULTS를 모두 끕니다."""
    for name in CALLS:
        CALLS[name] = 0
    POSTED.clear()

    # 앞 시연에서 켠 스위치가 다음 시연에 남지 않게 모두 끕니다.
    FAULTS["price_failures"] = 0
    FAULTS["price_format_changed"] = False
    FAULTS["board_down"] = False
    FAULTS["drop_response"] = False


# [외부 API] 경쟁사 가격: 앞의 몇 번은 연결이 끊깁니다(일시 오류). 형식이 바뀐 주에는 매번 실패합니다(영구 오류).
def fetch_prices():
    """경쟁사 가격 목록을 반환합니다. 앞의 price_failures번은 ConnectionError, price_format_changed면 KeyError를 냅니다."""
    CALLS["fetch_prices"] += 1
    if CALLS["fetch_prices"] <= FAULTS["price_failures"]:
        raise ConnectionError("가격 API 연결 끊김")

    # 응답에 "이번 주" 키가 없어진 경우입니다. 몇 번을 다시 불러도 같은 KeyError입니다.
    if FAULTS["price_format_changed"]:
        raise KeyError("이번 주")

    return COMPETITOR_PRICES


# [외부 API] 업계 뉴스: 이 교안에서는 늘 바로 응답합니다.
async def fetch_news():
    """업계 뉴스를 반환합니다."""
    CALLS["fetch_news"] += 1
    return NEWS


# [외부 API] 고객 리뷰: 응답 형식이 바뀌어 매번 실패합니다(영구 오류).
def fetch_reviews():
    """고객 리뷰를 반환합니다. 예전 형식의 키를 읽어서 매번 KeyError를 냅니다."""
    CALLS["fetch_reviews"] += 1
    response = REVIEW_API_RESPONSE

    # 새 형식에는 "reviews" 키가 없습니다. 몇 번을 다시 불러도 같은 KeyError입니다.
    return response["reviews"]


# [외부 API] 사내 게시판 게시: 외부 시스템의 상태를 바꾸는 부작용 함수입니다.
def post_briefing(text, key=None):
    """게시판에 글을 올리고 글 번호를 반환합니다. key가 같은 글이 이미 있으면 새로 올리지 않습니다."""
    CALLS["post_briefing"] += 1
    if FAULTS["board_down"]:
        raise ConnectionError("게시판 API 점검 중")

    # 멱등 키 확인: 같은 키로 올린 글이 있으면 그 글 번호를 그대로 돌려줍니다.
    if key is not None:
        for post in POSTED:
            if post["key"] == key:
                return post["id"]

    post_id = f"POST-{len(POSTED) + 1:03d}"
    POSTED.append({"id": post_id, "key": key, "text": text})

    # 글은 올라갔는데 응답만 한 번 끊기는 경우입니다. 호출한 쪽은 ConnectionError만 받아 게시가 실패한 줄 압니다.
    if FAULTS["drop_response"]:
        FAULTS["drop_response"] = False
        raise ConnectionError("게시 응답 끊김")

    return post_id

## 2. 병렬 분기의 부분 실패

### 2-1. 부분 실패와 격리

55일차의 `Send`로 가격·뉴스·리뷰 조사를 동시에 실행하면 세 조사는 **같은 단계**(superstep)에서 실행됩니다. LangGraph는 한 단계를 한 묶음으로 처리합니다. 그 단계의 작업 하나가 예외를 내면 아직 실행 중인 작업은 취소되고, 그 단계의 결과는 반영되지 않으며, 다음 노드(`synthesizer`)도 실행되지 않습니다.

**부분 실패 격리**는 작업 하나의 실패를 그 작업의 결과로 기록하고, 나머지 결과로 팀이 끝까지 가게 하는 방법입니다.

<img src="images/07_partial_failure.png" width="1050" alt="그대로 두면 리뷰 조사원의 KeyError 하나로 가격·뉴스 조사원의 자료까지 반영되지 않아 브리핑이 없고, 실패를 기록하면 가격·뉴스 자료는 findings에, 리뷰 실패 쪽지는 failures에 들어가 맨 아래에 수집하지 못한 자료: 고객 리뷰가 적힌 브리핑이 완성됩니다.">

리뷰 API 형식이 바뀐 주:

1. 리뷰 조사가 `KeyError`를 냅니다.
2. 조사 노드가 예외를 잡아 `failures`에 `{"reviews": "KeyError: 'reviews'"}`를 기록하고 정상 반환합니다.
3. 가격·뉴스 조사는 그대로 `findings`에 남습니다.
4. `synthesizer`와 `writer`가 가격·뉴스로 브리핑을 쓰고, "수집하지 못한 자료: 고객 리뷰"를 밝힙니다.

<img src="images/node_flow/l2_team_overview.png" width="1050" alt="planner가 가격·뉴스·리뷰 작업마다 Send로 researcher를 동시에 실행하고, 모두 끝나면 synthesizer와 writer가 차례로 실행됩니다.">

### 2-2. 브리핑 팀 State와 계획·종합·작성 노드

State와 조사 계획(`planner`), 종합(`synthesizer`), 작성(`writer`) 노드는 55일차 팀과 같은 방식이라 한 셀에 모았습니다. `findings`와 `failures`는 `merge_results` 리듀서로 작업 이름별로 모읍니다. 수집하지 못한 자료는 `synthesizer`에도 넘겨 추측하지 않게 하고, `writer`가 LLM에 맡기지 않고 코드로 브리핑 끝에 한 줄 붙입니다.

<img src="images/node_flow/l2_team_planner_synthesizer_writer.png" width="1050" alt="주간 브리핑 팀 그래프에서 지금 만드는 노드: planner, synthesizer, writer">

그림에서 주황색으로 표시한 노드 셋(`planner`, `synthesizer`, `writer`)을 아래 셀 하나에서 만듭니다. `researcher`는 이 절에서 실패 처리를 바꿔 가며 다루므로 2-3에서 따로 만듭니다.

In [ ]:
from langgraph.types import Send


# [리듀서] 키(작업 이름)별로 교체하는 dict 병합
def merge_results(old: dict, new: dict) -> dict:
    """기존 결과에 새 결과를 합칩니다. 같은 키는 새 값으로 바꿉니다."""
    merged = dict(old)
    merged.update(new)

    return merged


# [State] 주간 시장 브리핑 팀의 공유 데이터
class BriefingState(TypedDict):
    week: str  # 브리핑 주차
    tasks: list[str]  # planner: 조사 작업 이름
    findings: Annotated[dict[str, str], merge_results]  # researcher: 작업별 조사 메모
    failures: Annotated[dict[str, str], merge_results]  # researcher: 작업별 실패 내용
    summary: str  # synthesizer: 핵심 사항
    briefing: str  # writer: 게시할 브리핑
    post_id: str  # publisher(4절): 게시판 글 번호


# [보조 함수] 시작 입력 만들기
def make_input(week):
    """주차 하나로 시작 상태를 만듭니다. 결과 필드는 비워 둡니다."""
    return {"week": week, "tasks": [], "findings": {}, "failures": {}, "summary": "", "briefing": "", "post_id": ""}


# [노드] 조사 계획: 작업 셋은 매주 같으므로 LLM에 맡기지 않습니다.
def planner(state: BriefingState):
    """가격·뉴스·리뷰 조사 작업을 반환합니다."""
    return {"tasks": ["prices", "news", "reviews"]}


# [라우팅 함수] 작업마다 Send 만들기
def assign_research(state: BriefingState):
    """작업마다 Send 하나로 researcher에게 작업 이름만 넘깁니다."""
    return [Send("researcher", {"source": source}) for source in state["tasks"]]


# [보조 함수] LLM 컨텍스트 만들기: 조사 메모를 작업 이름과 함께 한 문자열로 묶습니다.
def format_findings(findings):
    """작업 이름과 조사 메모를 한 문자열로 묶습니다."""
    blocks = []
    for source, text in findings.items():
        # 한국어 작업 이름을 대괄호로 붙여 어느 조사에서 나온 메모인지 모델이 구분하게 합니다.
        blocks.append(f"[{SOURCE_NAMES[source]}]\n{text}")

    return "\n\n".join(blocks)


# [보조 함수] 수집 실패 목록: 실패한 작업의 한국어 이름만 적습니다. 예외 원문은 failures와 trace에만 남깁니다.
def failure_note(failures):
    """실패한 작업 이름을 쉼표로 이어 반환합니다. 실패가 없으면 '없음'입니다."""
    names = []
    for source in failures:
        names.append(SOURCE_NAMES[source])

    if not names:
        return "없음"
    return ", ".join(names)


# 종합 프롬프트: 수집하지 못한 자료에 대해 추측하지 않게 합니다.
synthesizer_prompt = ChatPromptTemplate.from_messages([
    ("system", "조사 메모를 AP-500 주간 브리핑의 핵심 사항 3~5개로 합칩니다. 메모에 없는 내용은 쓰지 마세요. "
     "수집하지 못한 자료가 있으면 그 자료에 대한 판단은 쓰지 말고 '이번 주 확인 못 함'으로 적으세요."),
    ("human", "우리 제품: {our_product}\n\n조사 메모:\n{findings}\n\n수집하지 못한 자료: {failures}"),
])
synthesizer_chain = synthesizer_prompt | llm


# [노드] 조사 종합
def synthesizer(state: BriefingState):
    """조사 메모와 수집 실패를 함께 넘겨 핵심 사항을 만듭니다."""
    # 성공한 메모만 findings에 있으므로, 빠진 자료는 failures로 따로 알려 줘야 모델이 추측하지 않습니다.
    summary = synthesizer_chain.invoke({
        "our_product": OUR_PRODUCT,
        "findings": format_findings(state["findings"]),
        "failures": failure_note(state["failures"]),
    })
    return {"summary": summary.text}


# 작성 프롬프트: 게시판 글 형식과 길이를 정합니다.
writer_prompt = ChatPromptTemplate.from_messages([
    ("system", "온결 마케팅팀 사내 게시판에 올릴 'AP-500 주간 시장 브리핑'을 씁니다. "
     "제목 한 줄, 핵심 사항 목록, 다음 주에 확인할 일 순서로 600자 이내로 쓰세요. 핵심 사항에 없는 내용은 쓰지 마세요."),
    ("human", "주차: {week}\n\n핵심 사항:\n{summary}"),
])
writer_chain = writer_prompt | llm


# [노드] 브리핑 작성
def writer(state: BriefingState):
    """핵심 사항으로 브리핑을 쓰고, 수집 실패가 있으면 코드로 끝에 한 줄을 붙입니다."""
    briefing = writer_chain.invoke({"week": state["week"], "summary": state["summary"]}).text

    # LLM이 빠뜨려도 실패 안내가 남도록 코드가 붙입니다.
    if state["failures"]:
        briefing += f"\n\n수집하지 못한 자료: {failure_note(state['failures'])} (수집 오류, 운영팀 확인 중)"

    return {"briefing": briefing}

### 2-3. 조사 노드 (`researcher`)

`Send`가 넘긴 `{"source": ...}`만 받아 작업 이름에 맞는 외부 API를 부르고, 받은 자료를 LLM으로 요약해 `findings`에 남깁니다. `fetch_news`가 async이므로 이 노드도 `async def`이고 LLM도 `ainvoke`로 부릅니다. 이 노드는 아직 실패를 격리하지 않습니다.

<img src="images/node_flow/l2_team_researcher.png" width="1050" alt="researcher: 작업 하나의 외부 API를 부르고 LLM으로 요약합니다. 작업 수만큼 동시에 실행됩니다.">

In [ ]:
# [보조 함수] 작업 이름으로 외부 API 부르기: 뉴스만 async라 await가 필요합니다.
async def fetch_source(source):
    """작업 이름에 맞는 외부 API를 불러 자료 목록을 반환합니다."""
    if source == "prices":
        return fetch_prices()
    if source == "news":
        return await fetch_news()
    return fetch_reviews()


# [보조 함수] LLM 컨텍스트 만들기: 자료 목록을 한 줄에 하나씩 JSON 문자열로 바꿉니다.
def format_items(items):
    """자료 목록을 줄마다 JSON 한 건인 문자열로 바꿉니다."""
    lines = []
    for item in items:
        # ensure_ascii=False: 한글을 \uXXXX로 바꾸지 않아야 모델이 회사·모델 이름을 그대로 읽습니다.
        lines.append(json.dumps(item, ensure_ascii=False))

    return "\n".join(lines)


# 조사원 프롬프트: 자료에 있는 값만 짧게 정리하게 합니다.
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "온결 마케팅팀의 시장 조사원입니다. 받은 {source} 자료를 공기청정기 AP-500 주간 브리핑에 쓸 메모 3줄 이내로 정리하세요. "
     "금액·날짜·이름은 자료 그대로 쓰고, 자료에 없는 내용은 쓰지 마세요."),
    ("human", "우리 제품: {our_product}\n\n자료:\n{items}"),
])
summary_chain = summary_prompt | llm


# [노드] 조사(격리 전): 외부 API의 예외를 그대로 올립니다.
async def researcher(payload):
    """작업 하나의 외부 API를 부르고 LLM 요약을 작업 이름 키로 findings에 남깁니다."""
    source = payload["source"]
    # 외부 API가 예외를 내면 여기서 그대로 올라가 같은 단계의 다른 작업까지 취소됩니다.
    items = await fetch_source(source)

    memo = await summary_chain.ainvoke({"source": SOURCE_NAMES[source], "our_product": OUR_PRODUCT, "items": format_items(items)})
    # 모델 응답은 메시지 객체라 .text로 글만 꺼내 State에 담습니다.
    return {"findings": {source: memo.text}}

### 2-4. 격리 없이 실행: 리뷰 하나의 예외로 팀 실행 전체가 실패

2-3의 `researcher`를 그대로 써서 팀을 연결하고 실행합니다. 리뷰 API는 스위치 없이 늘 `KeyError`를 내므로 켜는 실패는 없습니다.

In [ ]:
# 그래프 연결: planner -> researcher(작업 수만큼) -> synthesizer -> writer
builder = StateGraph(BriefingState)
builder.add_node("planner", planner)
builder.add_node("researcher", researcher)
builder.add_node("synthesizer", synthesizer)
builder.add_node("writer", writer)

builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", assign_research, ["researcher"])
builder.add_edge("researcher", "synthesizer")
builder.add_edge("synthesizer", "writer")
builder.add_edge("writer", END)

fragile_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(fragile_app.get_graph().draw_mermaid_png()))

In [ ]:
# 켜는 스위치 없음: 리뷰 API의 KeyError 하나만 일어납니다.
reset_demo()
partial_session = f"l3-partial-{run_stamp}"
partial_config = make_config(partial_session, partial_session, ["day56", "lesson03", "partial-failure", "v1-no-isolation"])

# 리뷰 조사 하나의 KeyError가 팀 실행 전체의 예외가 됩니다.
fragile_result = None
try:
    fragile_result = await fragile_app.ainvoke(make_input(WEEK), partial_config)
except KeyError as error:
    print("팀 실행 실패:", type(error).__name__, error)

print("브리핑 결과:", fragile_result)
print("trace:", trace_url(partial_config))

**확인할 점:** `팀 실행 실패: KeyError 'reviews'`가 출력되고 `브리핑 결과`는 `None`입니다. 가격·뉴스 API는 정상이었지만 그 조사 결과도 남지 않았습니다. 병렬 조사 하나의 예외가 팀 실행 전체를 실패시켰습니다.

### 2-5. 실패를 기록하는 조사 노드

조사 노드 안에서 외부 API 호출을 `try/except`로 감쌉니다. 두 종류를 다르게 처리합니다.

| 예외 | 처리 | 이유 |
|---|---|---|
| `ConnectionError` (일시 오류) | 다시 올림 | 노드의 `RetryPolicy`가 이 작업만 다시 시도하게 합니다 |
| 그 밖의 예외 (영구 오류) | `failures`에 기록하고 정상 반환 | 같은 단계의 다른 작업은 계속되고 다음 노드가 실행됩니다 |

In [ ]:
# [노드] 조사(격리 후): 영구 오류는 failures에 기록하고 정상 반환합니다.
async def researcher_safe(payload):
    """외부 API 영구 오류는 failures에 기록하고, 성공하면 LLM 요약을 findings에 남깁니다."""
    source = payload["source"]
    try:
        items = await fetch_source(source)
    except ConnectionError:
        # 일시 오류는 다시 올려야 RetryPolicy가 이 작업만 다시 시도합니다.
        raise
    except Exception as error:
        # 영구 오류는 이 작업의 결과로 기록합니다. 정상 반환이므로 같은 단계의 다른 작업은 계속됩니다.
        return {"failures": {source: f"{type(error).__name__}: {error}"}}

    memo = await summary_chain.ainvoke({"source": SOURCE_NAMES[source], "our_product": OUR_PRODUCT, "items": format_items(items)})
    return {"findings": {source: memo.text}}

In [ ]:
# researcher만 바꾸고, 일시 오류용 RetryPolicy를 붙입니다. 나머지 연결은 2-4와 같습니다.
# retry_on=ConnectionError라 모델 오류(openai 예외)는 노드에서 다시 시도하지 않고 SDK 재시도에만 맡깁니다.
builder = StateGraph(BriefingState)
builder.add_node("planner", planner)
builder.add_node("researcher", researcher_safe,
                 retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.5, retry_on=ConnectionError))
builder.add_node("synthesizer", synthesizer)
builder.add_node("writer", writer)

builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", assign_research, ["researcher"])
builder.add_edge("researcher", "synthesizer")
builder.add_edge("synthesizer", "writer")
builder.add_edge("writer", END)

isolated_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(isolated_app.get_graph().draw_mermaid_png()))

In [ ]:
# 켜는 스위치 없음: 2-4와 같은 조건(리뷰 API의 KeyError 하나)입니다.
reset_demo()
partial_config = make_config(partial_session, partial_session, ["day56", "lesson03", "partial-failure", "v2-isolated"])

# 리뷰 조사는 여전히 실패하지만 팀 실행은 끝까지 갑니다.
isolated_result = await isolated_app.ainvoke(make_input(WEEK), partial_config)

print("조사 메모가 있는 작업:", list(isolated_result["findings"]))
print("실패 기록:", isolated_result["failures"])
print("trace:", trace_url(partial_config))

In [ ]:
# 게시판에 올릴 브리핑입니다. 모델이 쓰므로 실행마다 문장이 다릅니다.
display(Markdown(isolated_result["briefing"]))

**확인할 점:**

- `조사 메모가 있는 작업`은 `['prices', 'news']`, `실패 기록`은 `{'reviews': "KeyError: 'reviews'"}`입니다.
- 브리핑 마지막 줄은 "수집하지 못한 자료: 고객 리뷰 (수집 오류, 운영팀 확인 중)"입니다.
- 핵심 사항에서 고객 리뷰는 "고객 리뷰는 이번 주 확인하지 못했다."처럼 확인하지 못했다는 말로만 나옵니다. 모델이 쓰는 문장이라 표현은 실행마다 다릅니다.

리뷰 조사만 실패로 기록되고, 팀은 가격·뉴스 자료로 브리핑까지 썼습니다.

### 2-6. Langfuse에서 두 실행 비교

Langfuse 화면 주소의 형식과 읽는 법은 교안 01의 5-4와 같습니다.

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces/<trace id>` (2-4에서 출력한 v1-no-isolation trace 주소)
- 누를 곳: 2-4에서 출력된 trace 주소(태그 v1-no-isolation), 상세 화면의 **Tree** 탭
- 보여 주는 것: v1은 팀 전체(briefing-team)가 Error이고 `researcher` 셋이 모두 Error입니다. 리뷰 조사는 KeyError로 실패했고, 함께 실행 중이던 가격·뉴스 조사는 그 순간 취소되어 같이 Error가 됩니다. 그 뒤 `synthesizer` 기록은 없습니다.

<img src="images/langfuse/l3_partial_failure_v1.png" width="1050" alt="Langfuse v1-no-isolation trace의 Tree에서 briefing-team과 researcher 세 개가 모두 Error로 표시되고 synthesizer가 없는 부분을 빨간 상자로 강조한 화면">

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces/<trace id>?observation=<항목 id>` (2-5에서 출력한 v2-isolated trace 주소에서 briefing-team을 누른 주소)
- 누를 곳: 2-5에서 출력된 trace 주소(태그 v2-isolated), 상세 화면의 **Tree** 탭, 맨 위 **briefing-team**
- 보여 주는 것: v2는 세 `researcher`가 모두 정상으로 끝나고 `synthesizer`가 이어집니다. 오른쪽 Output의 `failures`에 리뷰 조사의 KeyError가 보입니다.

<img src="images/langfuse/l3_partial_failure_v2.png" width="1050" alt="Langfuse v2-isolated trace에서 researcher 세 개 뒤에 synthesizer가 이어진 기록과, 팀 Output의 failures에 reviews KeyError가 든 줄을 빨간 상자로 강조한 화면">

### 2-7. 실패 뒤 진행 판단: 축소·보류·중단

실패를 기록하면 "어떤 자료가 빠져도 브리핑을 낼지"도 정해야 합니다. 2-5의 팀은 무엇이 실패하든 다음 단계로 갑니다. 리뷰가 빠진 브리핑은 내도 되지만, **이번 주 경쟁사 가격**이 빠진 시장 브리핑은 핵심이 빠진 글입니다. **진행 판단**은 실패를 격리한 뒤 이번 실행을 계속해도 되는지 정하는 단계입니다. 조사 자료를 필수 자료와 선택 자료로 나누고, 무엇이 실패했는지로 실행 결과(`outcome`)를 정합니다.

| 자료 | 구분 | 이유 |
|---|---|---|
| 경쟁사 가격 (`prices`) | 필수 자료 | 이번 주 가격 비교가 브리핑의 핵심입니다 |
| 업계 뉴스 (`news`), 고객 리뷰 (`reviews`) | 선택 자료 | 빠지면 그 부분을 "확인 못 함"으로 밝히면 됩니다 |

| 조사 결과 | `outcome` | 다음 단계 |
|---|---|---|
| 실패 없음 | `completed` (완료) | 그대로 `synthesizer -> writer` |
| 선택 자료만 실패 | `degraded` (축소 진행) | 누락을 밝히고 `synthesizer -> writer` |
| 필수 자료 실패 | `held` (발행 보류) | 브리핑을 쓰지 않고 끝냄. 사람이 확인 |
| 모든 조사 실패 | `failed` (중단) | 브리핑을 쓰지 않고 끝냄 |

<img src="images/12_progress_decision.png" width="1050" alt="가격(필수), 뉴스(선택), 리뷰(선택) 조사 결과를 보고 진행 판단이 네 갈래로 보내는 그림: 실패 없음이면 완료 completed로 브리핑 완성, 선택 자료만 실패면 축소 진행 degraded로 누락을 밝힌 브리핑, 필수 자료 실패면 발행 보류 held로 사람 확인, 모든 조사 실패면 중단 failed.">

판단은 LLM 없이 코드로 합니다. 같은 실패에는 늘 같은 결론이 나야 하고, 판단 근거를 그대로 남겨야 하기 때문입니다. 판단 노드 `decide_after_research`는 모든 `researcher`가 끝난 뒤 한 번 실행되어 `Command(goto=..., update={"outcome": ..., "outcome_reason": ...})`로 다음 노드와 판단 이유를 함께 반환합니다. 판단이 노드의 출력이므로 Langfuse trace에서 이 노드를 누르면 결론과 이유가 보이고, 최종 State에도 남습니다.

<img src="images/node_flow/l2_team_decision_overview.png" width="1050" alt="planner가 Send로 researcher를 동시에 실행하고, 모두 끝나면 decide_after_research가 completed·degraded면 synthesizer와 writer로, held·failed면 END로 보냅니다.">

2-2의 `BriefingState`를 상속해 판단 결과 두 필드만 더합니다. `planner`, `researcher_safe`, `synthesizer`, `writer`는 2-5의 것을 그대로 씁니다.

In [ ]:
# [State] 2-2의 브리핑 팀 State에 진행 판단 결과 두 필드를 더합니다.
class DecisionState(BriefingState):
    outcome: str  # decide_after_research: completed, degraded, held, failed 중 하나
    outcome_reason: str  # decide_after_research: 판단 이유


# 빠지면 브리핑을 낼 수 없는 필수 자료입니다. 나머지(news, reviews)는 선택 자료입니다.
REQUIRED_SOURCES = ["prices"]

**진행 판단 노드 (`decide_after_research`):** 모든 조사가 실패했는지를 먼저 봅니다. 모두 실패한 경우에는 필수 자료도 빠져 있으므로, 순서를 바꾸면 `failed`가 `held`로 잘못 나옵니다.

<img src="images/node_flow/l2_team_decision_decide.png" width="1050" alt="decide_after_research: 실패 기록을 보고 outcome과 이유를 남기며 synthesizer나 END로 보냅니다.">

In [ ]:
# [노드] 진행 판단: 실패 기록을 보고 다음 노드와 실행 결과를 정합니다. LLM을 부르지 않습니다.
def decide_after_research(state: DecisionState) -> Command[Literal["synthesizer", "__end__"]]:
    """실패한 조사로 completed, degraded, held, failed 중 하나를 정하고 다음 노드로 보냅니다."""
    failures = state["failures"]
    if not failures:
        return Command(goto="synthesizer", update={"outcome": "completed", "outcome_reason": "실패한 조사 없음"})

    # 모든 조사가 실패하면 쓸 자료가 없습니다.
    if len(failures) == len(state["tasks"]):
        return Command(goto=END, update={
            "outcome": "failed",
            "outcome_reason": f"모든 조사 실패: {failure_note(failures)}",
        })

    # 필수 자료가 빠지면 브리핑을 쓰지 않고 끝냅니다. synthesizer와 writer의 모델 호출도 하지 않습니다.
    for source in REQUIRED_SOURCES:
        if source in failures:
            return Command(goto=END, update={
                "outcome": "held",
                "outcome_reason": f"필수 자료 실패: {SOURCE_NAMES[source]}, 발행 보류 후 사람 확인",
            })

    # 선택 자료만 빠졌으면 진행합니다. writer가 브리핑 끝에 수집하지 못한 자료를 붙입니다.
    return Command(goto="synthesizer", update={
        "outcome": "degraded",
        "outcome_reason": f"선택 자료 실패: {failure_note(failures)}, 누락을 밝히고 진행",
    })

In [ ]:
# researcher와 synthesizer 사이에 판단 노드를 넣습니다. 나머지 노드와 재시도 규칙은 2-5와 같습니다.
builder = StateGraph(DecisionState)
builder.add_node("planner", planner)
builder.add_node("researcher", researcher_safe,
                 retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.5, retry_on=ConnectionError))
builder.add_node("decide_after_research", decide_after_research)
builder.add_node("synthesizer", synthesizer)
builder.add_node("writer", writer)

builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", assign_research, ["researcher"])
builder.add_edge("researcher", "decide_after_research")
# decide_after_research 뒤는 Command가 정하므로 엣지를 적지 않습니다.
builder.add_edge("synthesizer", "writer")
builder.add_edge("writer", END)

decision_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(decision_app.get_graph().draw_mermaid_png()))

**시나리오 A: 선택 자료만 실패.** 2-5와 같은 조건(리뷰 API의 `KeyError` 하나)으로 실행합니다.

In [ ]:
# 켜는 스위치 없음: 리뷰 API의 KeyError 하나만 일어납니다.
reset_demo()
decision_session = f"l3-decision-{run_stamp}"
decision_config = make_config(decision_session, decision_session, ["day56", "lesson03", "progress-decision", "v1-degraded"])

degraded_result = await decision_app.ainvoke(make_input(WEEK), decision_config)

print("outcome:", degraded_result["outcome"], "| 이유:", degraded_result["outcome_reason"])
print("브리핑 마지막 줄:", degraded_result["briefing"].splitlines()[-1])
print("trace:", trace_url(decision_config))

**시나리오 B: 필수 자료 실패.** 가격 API의 응답 형식이 바뀐 주를 만듭니다. `price_format_changed` 스위치를 켜면 `fetch_prices`가 `KeyError`를 냅니다. `ConnectionError`가 아니므로 `researcher_safe`는 재시도 없이 `failures`에 기록합니다.

In [ ]:
reset_demo()
# 이 시연에서 켜는 실패: 가격 API 응답 형식 변경(영구 오류). 리뷰 API의 KeyError도 그대로입니다.
FAULTS["price_format_changed"] = True
decision_config = make_config(decision_session, decision_session, ["day56", "lesson03", "progress-decision", "v2-held"])

held_result = await decision_app.ainvoke(make_input(WEEK), decision_config)

print("outcome:", held_result["outcome"], "| 이유:", held_result["outcome_reason"])
print("실패 기록:", held_result["failures"])
# synthesizer와 writer가 실행되지 않았으면 make_input의 빈 문자열이 그대로 남습니다.
print("summary:", repr(held_result["summary"]), "| briefing:", repr(held_result["briefing"]))
print("trace:", trace_url(decision_config))

**확인할 점:**

- A: `outcome`이 `degraded`이고, 브리핑 마지막 줄이 "수집하지 못한 자료: 고객 리뷰 (수집 오류, 운영팀 확인 중)"입니다.
- B: `outcome`이 `held`이고, `실패 기록`에 `prices`(`KeyError: '이번 주'`)와 `reviews`가 있으며, `summary`와 `briefing`이 `''`입니다.

필수 자료인 가격이 빠지면 판단 노드가 `synthesizer`와 `writer`를 실행하지 않고 끝냅니다.

`held`와 `failed`는 예외가 아니라 정상 종료입니다. 교안 02 3-3의 대기 엣지처럼 예외 알림으로는 잡히지 않으므로, 실행이 끝나면 `outcome`을 보고 `held`면 담당자에게 알리거나 51일차의 `interrupt()`로 사람의 결정을 기다립니다.

**Langfuse에서 확인**

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces/<trace id>?observation=<항목 id>` (위에 출력된 v2-held trace 주소에서 decide_after_research를 누른 주소)
- 누를 곳: 위에 출력된 v2-held trace 주소, 상세 화면의 **Tree** 탭, **decide_after_research**
- 보여 주는 것: 판단 노드의 Input `failures`에 `prices`가 있고, Output `update`에 `outcome`(held)과 `outcome_reason`이 있습니다. `goto`가 `__end__`라 그 뒤에 `synthesizer`·`writer` 기록이 없습니다. v1-degraded trace에서는 같은 노드 뒤에 `synthesizer`와 `writer`가 이어집니다.

<img src="images/langfuse/l3_progress_held.png" width="1050" alt="Langfuse v2-held trace의 Tree에서 decide_after_research 노드를 고르고, 오른쪽 Input의 failures(prices KeyError), Output의 outcome(held)·outcome_reason, goto(__end__)를 빨간 상자로 강조한 화면">

## 3. 폴백 경로: `error_handler`

**폴백 경로**는 노드가 재시도까지 다 실패했을 때 대신 실행할 노드로 가는 길입니다. LangGraph 1.2부터 `add_node(..., error_handler=함수)`로 노드마다 붙일 수 있습니다. 실행 순서는 시도, `RetryPolicy` 재시도 소진, 그다음 핸들러입니다. 이 버전(1.2.9)에서는 같은 단계에 다른 작업이 있으면 핸들러가 실행된 뒤에도 예외가 다시 올라오므로, 병렬 분기가 아닌 **단독 노드**에만 씁니다. 병렬 분기는 2-5처럼 노드 안에서 기록합니다.

가격 API가 한동안 내려가 있는 상황:

1. 이번 주 가격 조사 노드 `collect_this_week`가 첫 시도와 재시도 1번 모두 `ConnectionError`로 실패합니다. 실패한 시도는 끝난 기록(updates)을 남기지 않으므로 `stream_trail`의 방문 순서에 `collect_this_week`가 나오지 않습니다.
2. 핸들러가 실패 정보(`NodeError`)를 받아 `Command(goto="use_last_week")`로 지난주 가격을 쓰는 노드로 보냅니다.
3. `use_last_week`가 지난주 가격을 넣고, `make_report`가 "지난주 가격 사용"을 밝힌 가격 보고를 만듭니다.

<img src="images/node_flow/l2_fallback_overview.png" width="1050" alt="collect_this_week가 성공하면 make_report로 가고, 재시도까지 실패하면 error_handler가 use_last_week로 보낸 뒤 make_report로 갑니다.">

핸들러 함수는 `(state, error: NodeError)`를 받습니다. `NodeError` 타입 힌트를 단 인자에 실패한 노드 이름(`error.node`)과 예외(`error.error`)가 들어옵니다. 핸들러는 **반드시 `Command(goto=...)`로 다음에 실행할 노드를 정합니다.** 핸들러가 실행되면 원래 노드의 고정 엣지(`collect_this_week -> make_report`)는 쓰이지 않기 때문입니다.

In [ ]:
from langgraph.errors import NodeError

# 지난주 가격: 회사별 지난주 값만 남긴 목록입니다.
LAST_WEEK_PRICES = []
for item in COMPETITOR_PRICES:
    LAST_WEEK_PRICES.append({"회사": item["회사"], "모델": item["모델"], "가격": item["지난주"]})


# [State] 가격 보고 그래프
class PriceReportState(TypedDict):
    prices: list[dict]  # collect_this_week 또는 use_last_week: 보고에 쓸 가격
    price_note: str  # collect_this_week 또는 핸들러: 어떤 가격인지, 폴백했다면 그 이유
    report: str  # make_report: 가격 보고


# [노드] 이번 주 가격 조사: 지난주 가격(LAST_WEEK_PRICES)과 같은 {회사, 모델, 가격} 모양으로 바꿔 담습니다.
def collect_this_week(state: PriceReportState):
    """이번 주 경쟁사 가격을 prices에 넣습니다."""
    prices = []
    for item in fetch_prices():
        prices.append({"회사": item["회사"], "모델": item["모델"], "가격": item["이번 주"]})

    return {"prices": prices, "price_note": "이번 주 가격"}


# [핸들러] 재시도 소진 뒤 실행: 실패 이유를 남기고 지난주 가격 노드로 보냅니다.
def price_fallback(state: PriceReportState, error: NodeError) -> Command[Literal["use_last_week"]]:
    """실패 정보를 price_note에 남기고 use_last_week로 이동합니다."""
    # 보고를 읽는 사람이 폴백 사실을 알도록 실패한 노드와 예외를 함께 적습니다.
    note = f"{error.node} 실패({type(error.error).__name__}: {error.error}), 지난주 가격 사용"
    return Command(goto="use_last_week", update={"price_note": note})


# [노드] 지난주 가격으로 대체
def use_last_week(state: PriceReportState):
    """지난주 가격을 prices에 넣습니다."""
    return {"prices": LAST_WEEK_PRICES}


# [노드] 가격 보고: 어떤 가격을 썼는지 첫 줄에 밝힙니다.
def make_report(state: PriceReportState):
    """가격 출처와 최저가 경쟁 제품을 한 문장으로 만듭니다."""
    # 첫 제품을 최저가로 두고 하나씩 비교합니다.
    cheapest = state["prices"][0]
    for item in state["prices"]:
        # 지금까지 찾은 제품보다 싸면 바꿉니다.
        if item["가격"] < cheapest["가격"]:
            cheapest = item

    return {"report": f"[{state['price_note']}] 최저가 경쟁 제품: {cheapest['회사']} {cheapest['모델']} {cheapest['가격']:,}원"}

In [ ]:
builder = StateGraph(PriceReportState)
# 첫 시도를 포함한 2번이 모두 실패하면 price_fallback이 실행됩니다.
builder.add_node("collect_this_week", collect_this_week,
                 retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.5, retry_on=ConnectionError),
                 error_handler=price_fallback)
builder.add_node("use_last_week", use_last_week)
builder.add_node("make_report", make_report)

builder.add_edge(START, "collect_this_week")
builder.add_edge("collect_this_week", "make_report")
builder.add_edge("use_last_week", "make_report")
builder.add_edge("make_report", END)

fallback_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(fallback_app.get_graph().draw_mermaid_png()))

In [ ]:
reset_demo()
# 이 시연에서 켜는 실패: 가격 API 앞 5번 연결 끊김. 재시도 2번으로는 회복되지 않습니다.
FAULTS["price_failures"] = 5
fallback_session = f"l3-fallback-{run_stamp}"
fallback_config = make_config(fallback_session, fallback_session, ["day56", "lesson03", "fallback", "v1-command"])

trail = []
fallback_result = stream_trail(fallback_app, {"prices": [], "price_note": "", "report": ""}, fallback_config, trail)

print("방문 순서:", " -> ".join(trail))
print("가격 API 호출 수:", CALLS["fetch_prices"])
print("가격 보고:", fallback_result["report"])

**확인할 점:** `가격 API 호출 수`가 2이고, `방문 순서`는 `__error_handler__collect_this_week -> use_last_week -> make_report`입니다. `가격 보고`는 "[collect_this_week 실패(ConnectionError: 가격 API 연결 끊김), 지난주 가격 사용]"으로 시작합니다. 두 시도가 모두 실패한 뒤 핸들러가 `use_last_week`로 보내 지난주 가격으로 보고를 만들었습니다.

**함정: 핸들러가 dict만 반환하면.** 핸들러가 `Command` 대신 `{"price_note": ...}`만 반환하면 State는 바뀌지만 다음 노드가 정해지지 않아 그래프가 거기서 끝납니다. 고정 엣지 `collect_this_week -> make_report`도 실행되지 않습니다.

In [ ]:
# [핸들러] 잘못된 예: 다음 노드를 정하지 않고 State만 바꿉니다.
def price_fallback_dict(state: PriceReportState, error: NodeError):
    """실패 이유만 price_note에 남깁니다."""
    return {"price_note": f"{error.node} 실패, 지난주 가격 사용"}


builder = StateGraph(PriceReportState)
builder.add_node("collect_this_week", collect_this_week,
                 retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.5, retry_on=ConnectionError),
                 error_handler=price_fallback_dict)
builder.add_node("use_last_week", use_last_week)
builder.add_node("make_report", make_report)

builder.add_edge(START, "collect_this_week")
builder.add_edge("collect_this_week", "make_report")
builder.add_edge("use_last_week", "make_report")
builder.add_edge("make_report", END)

dict_handler_app = builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(dict_handler_app.get_graph().draw_mermaid_png()))

In [ ]:
reset_demo()
# 이 시연에서 켜는 실패: 위와 같은 가격 API 앞 5번 연결 끊김
FAULTS["price_failures"] = 5
fallback_config = make_config(fallback_session, fallback_session, ["day56", "lesson03", "fallback", "v2-dict"])

# 예외는 없지만 make_report가 실행되지 않아 report가 빈 문자열로 남습니다.
trail = []
dict_result = stream_trail(dict_handler_app, {"prices": [], "price_note": "", "report": ""}, fallback_config, trail)

print("방문 순서:", " -> ".join(trail))
print("price_note:", dict_result["price_note"], "| report:", repr(dict_result["report"]))

**확인할 점:** `방문 순서`가 `__error_handler__collect_this_week` 하나에서 끝납니다. `price_note`는 "collect_this_week 실패, 지난주 가격 사용"으로 바뀌었지만 `report`는 `''`입니다. 핸들러가 다음 노드를 정하지 않아 `make_report`가 실행되지 않았고, 예외 없이 끝났으므로 `report`가 비었는지 직접 확인해야 알 수 있습니다.

## 4. 멈춘 곳에서 이어 하기

### 4-1. 게시 단계를 붙이고 체크포인터 달기

**이어 하기**(재개)는 실패한 실행을 처음부터 반복하지 않고, 실패한 지점부터 다시 실행하는 것입니다. 51일차의 체크포인터를 달고 같은 `thread_id`로 `invoke(None, config)`를 부르면 됩니다. 입력 자리의 `None`은 "새 입력 없이 저장된 상태에서 계속"이라는 뜻입니다. 조사와 작성처럼 LLM 호출이 많은 단계를 다시 하지 않으므로 비용과 시간이 줄어듭니다.

<img src="images/08_resume_checkpoint.png" width="1050" alt="planner, researcher x3, synthesizer, writer 정거장마다 체크포인트 깃발이 꽂혀 있고 publisher가 ConnectionError로 멈춥니다. 같은 thread_id의 invoke(None, config) 화살표가 앞의 네 정거장은 다시 안 하고 publisher로 바로 들어가 게시에 성공합니다.">

게시판 API가 점검 중인 주:

1. 조사·종합·작성까지 끝났습니다. 단계마다 체크포인트가 저장됩니다.
2. `publisher`가 게시판 API 점검으로 `ConnectionError`를 냅니다. `publisher`에는 재시도를 걸지 않았으므로 바로 실패로 끝납니다.
3. 점검이 끝난 뒤 같은 `thread_id`로 `invoke(None, config)`를 부릅니다.
4. `publisher`만 다시 실행됩니다. 조사 API와 LLM은 다시 불리지 않습니다.

2-5 팀 뒤에 `publisher`를 붙이고, `compile(checkpointer=InMemorySaver())`로 체크포인터를 답니다.

<img src="images/node_flow/l2_publish_overview.png" width="1050" alt="2-5 팀의 writer 뒤에 publisher가 붙어 게시판에 브리핑을 올리고 END로 끝납니다.">

<img src="images/node_flow/l2_publish_publisher.png" width="1050" alt="publisher: writer가 쓴 브리핑을 게시판에 올리고 글 번호를 post_id에 남깁니다.">

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver


# [노드] 게시: 외부 시스템의 상태를 바꾸는 단계라 팀의 맨 끝에 둡니다.
def publisher(state: BriefingState):
    """브리핑을 사내 게시판에 올리고 글 번호를 반환합니다."""
    return {"post_id": post_briefing(state["briefing"])}


builder = StateGraph(BriefingState)
builder.add_node("planner", planner)
builder.add_node("researcher", researcher_safe,
                 retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.5, retry_on=ConnectionError))
builder.add_node("synthesizer", synthesizer)
builder.add_node("writer", writer)
# publisher에는 재시도를 걸지 않습니다. 몇 초 안에 끝나지 않는 장애를 이어 하기로 처리하는 시연입니다.
builder.add_node("publisher", publisher)

builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", assign_research, ["researcher"])
builder.add_edge("researcher", "synthesizer")
builder.add_edge("synthesizer", "writer")
builder.add_edge("writer", "publisher")
builder.add_edge("publisher", END)

# 체크포인터가 있어야 실패한 실행의 상태가 남고 이어 할 수 있습니다.
publish_app = builder.compile(checkpointer=InMemorySaver())

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(publish_app.get_graph().draw_mermaid_png()))

In [ ]:
reset_demo()
# 이 시연에서 켜는 실패: 게시판 API 점검 중. 원인을 고치기 전까지 게시는 계속 실패합니다.
FAULTS["board_down"] = True

# 이어 하기도 이 config를 그대로 씁니다. 같은 langfuse_session_id라 두 실행이 한 세션에 묶이고, 같은 thread_id라 체크포인터가 멈춘 곳을 찾습니다.
resume_session = f"l3-resume-{run_stamp}"
resume_config = make_config(resume_session, resume_session, ["day56", "lesson03", "resume"])

try:
    await publish_app.ainvoke(make_input(WEEK), resume_config)
except ConnectionError as error:
    print("팀 실행 실패:", type(error).__name__, error)

print("외부 API 호출 수:", CALLS)
print("실패한 실행 trace:", trace_url(resume_config))

### 4-2. 멈춘 곳 읽기

`get_state(config)`는 그 `thread_id`의 마지막 체크포인트를 돌려줍니다.

| 필드 | 뜻 | 이번 실행에서 볼 것 |
|---|---|---|
| `next` | 다음에 실행할 노드 이름 튜플 | `('publisher',)` |
| `tasks` | 그 단계의 작업 목록. 작업마다 `name`, `error` | `publisher` 작업의 오류 |
| `values` | 저장된 State | `briefing`이 채워져 있음 |

`task.error`는 타입 힌트와 달리 **문자열**로 옵니다(예: `"ConnectionError('게시판 API 점검 중')"`). `isinstance(task.error, ConnectionError)`처럼 예외 클래스로 가르면 늘 거짓이므로 `if task.error:`로 있는지만 보고 문자열로 읽습니다.

In [ ]:
snapshot = publish_app.get_state(resume_config)

print("다음 노드:", snapshot.next)

# 실패한 작업만 이름과 오류를 출력합니다.
for task in snapshot.tasks:
    if task.error:
        print("실패한 작업:", task.name, "| 오류:", task.error, "| 오류 값의 타입:", type(task.error).__name__)

print("저장된 브리핑 글자 수:", len(snapshot.values["briefing"]), "| 실패 기록:", snapshot.values["failures"])

### 4-3. 원인 해결 뒤 이어 하기

게시판 점검이 끝났다고 보고 스위치를 끈 뒤, 같은 config로 `ainvoke(None, ...)`를 부릅니다.

In [ ]:
# 원인 해결: 게시판 API가 다시 열렸습니다.
FAULTS["board_down"] = False

# 입력 None: 새로 시작하지 않고 저장된 상태의 다음 노드(publisher)부터 실행합니다.
resumed = await publish_app.ainvoke(None, resume_config)

print("게시 글 번호:", resumed["post_id"], "| 게시판 글 수:", len(POSTED))
print("외부 API 호출 수:", CALLS)
print("이어 하기 trace:", trace_url(resume_config))

**확인할 점:** `외부 API 호출 수`에서 `fetch_prices`, `fetch_news`, `fetch_reviews`는 4-1과 같은 1이고, `post_briefing`만 1에서 2로 늘었습니다. `게시 글 번호`는 `POST-001`, `게시판 글 수`는 1입니다. 이어 하기가 조사 노드를 다시 실행하지 않고 실패한 `publisher`만 다시 실행했습니다.

**Langfuse에서 확인**

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces` (목록의 Session ID 값을 누르면 세션 화면 `https://cloud.langfuse.com/project/<프로젝트 ID>/sessions/<세션 id>`)
- 누를 곳: 왼쪽 메뉴 **Tracing**, 오른쪽 위 **Columns**에서 Status·Session ID 켜기, 위쪽 검색바에 `sessionId:l3-resume-...`(이번 세션 이름) 입력
- 보여 주는 것: 오류로 멈춘 실행과 이어 하기 실행이 trace 두 개로 남고, 두 실행이 같은 config의 `langfuse_session_id`를 썼으므로 한 세션에 함께 나옵니다. 빨간 상자 안의 Status 열에서 먼저 실행한 trace는 Error, 이어 하기 trace는 Default입니다. 이어 하기 trace는 publisher 하나만 실행하므로 Latency가 거의 0초입니다. 행을 눌러 Tree 탭을 열면 앞 trace는 `publisher`에서 Error로 끝났고, 뒤 trace에는 `publisher` 하나만 있습니다.

<img src="images/langfuse/l3_session_resume.png" width="1050" alt="Langfuse Tracing 목록을 검색바의 sessionId 조건으로 거른 화면에서 그 조건, Status 열(Error, Default), Session ID 열을 빨간 상자로 강조한 화면">

### 4-4. 멱등 게시: 두 번 실행돼도 한 번만

**멱등성**은 같은 요청을 여러 번 실행해도 결과가 한 번 실행한 것과 같은 성질입니다. **멱등 키**는 "이 일은 이미 했다"를 가리는 고유 값입니다. 게시할 때 키를 함께 보내고, 같은 키의 글이 이미 있으면 새로 올리지 않습니다.

<img src="images/09_idempotent_post.png" width="1050" alt="게시는 됐는데 응답이 끊겨 이어 하기로 다시 게시하는 상황입니다. 키 없음 쪽은 호출 2번에 게시판에 주간 브리핑이 두 장(글 2개) 붙어 직원이 헷갈려 하고, 멱등 키 쪽은 같은 호출 2번이지만 게시 키 목록에서 weekly-2026-41을 확인한 직원이 이미 게시함, 건너뜀이라고 말해 주간 브리핑이 한 장(글 1개)만 있습니다.">

재시도와 이어 하기는 실패한 노드를 **다시 실행**합니다. 병렬 단계에서는 이미 끝난 작업은 다시 실행되지 않지만, 다른 작업의 실패로 취소된 작업은 이어 하기 때 처음부터 다시 실행됩니다. 검색처럼 읽기만 하는 노드는 여러 번 실행돼도 괜찮지만, 게시·메일 발송·결제처럼 외부 시스템의 상태를 바꾸는 노드는 두 번 실행되면 그 변경도 두 번 일어납니다.

1. `publisher`가 브리핑을 게시판에 올립니다. 글은 올라갔는데 응답이 끊겨 `ConnectionError`가 납니다.
2. LangGraph는 `publisher`가 실패했다고 기록합니다. 글이 올라갔는지는 모릅니다.
3. 이어 하기로 `publisher`가 다시 실행되어 같은 브리핑이 한 번 더 올라갑니다.
4. 키 `weekly-2026-41`을 함께 보냈다면 3번의 게시는 이미 올린 글 번호만 돌려받고 새로 올리지 않습니다.

브리핑은 주마다 한 번이므로 `weekly-{주차}`를 키로 씁니다. 실행 단위로 막으려면 `thread_id`를 키에 넣을 수도 있지만, 주차 키는 같은 주에 팀을 새로 한 번 더 돌린 경우까지 막습니다. 정정본을 다시 올려야 하면 키에 버전 번호를 붙입니다(예: `weekly-2026-41-v2`).

LLM 없이 게시 노드 하나만 있는 그래프로, 멱등 키가 있을 때와 없을 때 게시판 글 수를 비교합니다. 게시판 응답이 한 번 끊기도록 `drop_response` 스위치를 켭니다.

In [ ]:
# [State] 게시 그래프
class PostState(TypedDict):
    week: str
    briefing: str
    post_id: str


# [노드] 키 없이 게시
def publish_plain(state: PostState):
    """브리핑을 키 없이 게시합니다."""
    return {"post_id": post_briefing(state["briefing"])}


# [노드] 멱등 키로 게시: 같은 주차의 브리핑은 한 번만 올라갑니다.
def publish_once(state: PostState):
    """주차로 만든 키를 함께 보내 게시합니다."""
    key = f"weekly-{state['week']}"
    return {"post_id": post_briefing(state["briefing"], key=key)}


# 게시 노드 하나짜리 그래프 두 개를 만듭니다. 이어 하기를 위해 둘 다 체크포인터를 답니다.
builder = StateGraph(PostState)
builder.add_node("publisher", publish_plain)
builder.add_edge(START, "publisher")
builder.add_edge("publisher", END)
plain_post_app = builder.compile(checkpointer=InMemorySaver())

builder = StateGraph(PostState)
builder.add_node("publisher", publish_once)
builder.add_edge(START, "publisher")
builder.add_edge("publisher", END)
once_post_app = builder.compile(checkpointer=InMemorySaver())

# 두 그래프의 모양은 같습니다. 하나만 그림으로 확인합니다.
display(Image(once_post_app.get_graph().draw_mermaid_png()))

In [ ]:
# [보조 함수] 첫 실행이 실패하면 같은 config로 멈춘 단계부터 한 번 다시 실행합니다.
def post_with_resume(app, config):
    """게시 그래프를 실행하고, 실패하면 invoke(None)으로 멈춘 단계부터 다시 실행한 뒤 게시판 글 수를 반환합니다."""
    try:
        app.invoke({"week": WEEK, "briefing": "AP-500 주간 시장 브리핑 (시연용 본문)", "post_id": ""}, config)
    except ConnectionError as error:
        # 응답만 끊긴 실패입니다. 글은 이미 올라가 있지만 그래프는 publisher를 실패로 기록합니다.
        print("첫 실행 실패:", error)
        # 같은 thread_id로 이어 하면 실패한 publisher가 다시 실행됩니다.
        app.invoke(None, config)

    return len(POSTED)


reset_demo()
# 이 시연에서 켜는 실패: 게시는 되지만 응답이 한 번 끊김
FAULTS["drop_response"] = True
plain_session = f"l3-post-plain-{run_stamp}"

# 키 없이: 응답이 끊긴 첫 게시와 이어 하기의 게시가 둘 다 남습니다.
plain_count = post_with_resume(plain_post_app, make_config(plain_session, plain_session, ["day56", "lesson03", "idempotent", "v1-no-key"]))
print("키 없이 게시판 글 수:", plain_count, "| post_briefing 호출 수:", CALLS["post_briefing"])

In [ ]:
reset_demo()
# 이 시연에서 켜는 실패: 위와 같이 게시는 되지만 응답이 한 번 끊김
FAULTS["drop_response"] = True
once_session = f"l3-post-once-{run_stamp}"

# 멱등 키로: 두 번째 호출은 같은 키의 글 번호를 돌려주고 새로 올리지 않습니다.
once_count = post_with_resume(once_post_app, make_config(once_session, once_session, ["day56", "lesson03", "idempotent", "v2-key"]))
print("멱등 키로 게시판 글 수:", once_count, "| post_briefing 호출 수:", CALLS["post_briefing"])

**확인할 점:** 두 실행 모두 `post_briefing 호출 수`는 2입니다. `게시판 글 수`는 키 없이 2, 멱등 키로 1입니다. 이어 하기로 `publisher`가 다시 실행되어도 멱등 키가 두 번째 게시를 막았습니다.

체크포인터는 **성공한 노드를 다시 실행하지 않는 것**까지만 보장합니다. 실패한 노드 안에서 이미 일어난 부작용은 노드가 직접 막아야 합니다. 병렬 단계에서 취소된 작업도 이어 하기 때 다시 실행되므로, 병렬 노드 안의 부작용에도 같은 키가 필요합니다.

### 4-5. 과거 결정으로 돌아가 다시 실행하기 (fork)

멀티에이전트 팀의 결정(어느 워커에게 넘길지, 어느 채널로 보낼지)은 LLM이 내리므로 같은 입력에도 실행마다 달라질 수 있습니다. 잘못된 결정 하나 때문에 처음부터 다시 실행하면 앞 단계의 LLM 호출을 모두 다시 하고, 다시 해도 같은 결정이 나온다는 보장이 없습니다.

**타임 트래블**은 체크포인트 기록에서 과거 시점으로 돌아가는 것입니다. 51일차에는 `get_state_history`로 기록을 읽기만 했습니다. 이번에는 잘못된 결정 **직후**의 체크포인트에서 결정 값만 고친 새 체크포인트를 만들고 거기서부터 실행합니다. 이렇게 기록이 갈라져 나온 실행을 **fork**(갈래)라고 합니다.

<img src="images/11_time_travel_fork.png" width="1050" alt="route_channel이 channel: email_all로 잘못 고른 직후의 체크포인트에서 update_state로 channel: board로 고친 새 갈래를 만들고 invoke(None)으로 post_board까지 실행하는 그림. route_channel은 다시 실행되지 않고, 원래 갈래의 send_email_all로 이미 나간 메일은 그대로 남습니다.">

1. `get_state_history(config)`로 체크포인트를 최신부터 읽어 잘못된 결정 직후의 체크포인트를 찾습니다. 그 체크포인트의 `next`가 잘못 고른 다음 노드입니다.
2. `update_state(그 체크포인트의 config, 고칠 값, as_node="결정 노드")`로 값을 고친 새 체크포인트를 만듭니다. `as_node`는 이 값을 어느 노드가 쓴 것으로 칠지 정합니다. 그 노드 뒤의 엣지가 다시 계산되어 `next`가 바뀝니다.
3. `update_state`가 돌려준 config로 `invoke(None, config)`를 부르면 그 지점부터 실행합니다. 결정 노드와 그 앞 노드는 다시 실행되지 않습니다.

브리핑 배포 단계로 확인합니다. `route_channel` 노드가 완성된 브리핑을 보낼 채널을 정하는데, 게시판(`board`)이어야 할 것을 전체 메일(`email_all`)로 잘못 골랐다고 합니다. LLM의 결정 대신 늘 `email_all`을 고르는 코드로 잘못된 결정을 재현합니다.

**결정은 State에 쓰고 조건 분기가 읽게 합니다.** 결정 노드가 `Command(goto=...)`로만 다음 노드를 정하면, `update_state`로 값을 고쳐도 다음 노드가 다시 계산되지 않아 `next`가 빈 튜플이 됩니다. 나중에 고칠 수 있어야 하는 결정은 State 필드로 남깁니다. 2-7의 `decide_after_research`는 `outcome`을 State에 남기지만 다음 노드는 `Command`로 정하므로 이 방법으로는 고칠 수 없습니다. 고칠 수 있게 하려면 `outcome`을 읽는 조건 분기로 바꿉니다.

배포 그래프의 State·노드·연결은 55일차와 같은 방식이라 한 셀에서 그래프까지 만듭니다.

In [ ]:
# 노드별 실행 횟수: 갈래를 실행한 뒤 어느 노드가 다시 실행되었는지 셉니다.
DELIVERY_RUNS = {"route_channel": 0, "post_board": 0, "send_email_all": 0}


# [State] 브리핑 배포 그래프
class DeliveryState(TypedDict):
    briefing: str
    channel: str  # route_channel: 보낼 채널 (board 또는 email_all)
    sent: Annotated[list[str], operator.add]  # 채널 노드: 실제로 보낸 채널


# [노드] 채널 결정: LLM의 잘못된 결정을 재현하려고 늘 email_all을 고릅니다.
def route_channel(state: DeliveryState):
    """브리핑을 보낼 채널을 channel에 씁니다."""
    DELIVERY_RUNS["route_channel"] += 1
    return {"channel": "email_all"}


# [라우팅 함수] channel 값으로 채널 노드를 고릅니다. update_state로 channel을 고치면 이 함수가 다시 판단합니다.
def pick_channel(state: DeliveryState):
    """channel이 board면 post_board, 아니면 send_email_all을 반환합니다."""
    if state["channel"] == "board":
        return "post_board"
    return "send_email_all"


# [노드] 게시판 게시와 전체 메일 발송: 실제 발송 대신 실행 횟수와 채널 이름만 남깁니다.
def post_board(state: DeliveryState):
    """게시판에 올렸다고 기록합니다."""
    DELIVERY_RUNS["post_board"] += 1
    return {"sent": ["board"]}


def send_email_all(state: DeliveryState):
    """전체 메일을 보냈다고 기록합니다."""
    DELIVERY_RUNS["send_email_all"] += 1
    return {"sent": ["email_all"]}


# 그래프 연결: route_channel 뒤에서 pick_channel이 channel 값으로 채널 노드를 고릅니다.
builder = StateGraph(DeliveryState)
builder.add_node("route_channel", route_channel)
builder.add_node("post_board", post_board)
builder.add_node("send_email_all", send_email_all)

builder.add_edge(START, "route_channel")
builder.add_conditional_edges("route_channel", pick_channel, ["post_board", "send_email_all"])
builder.add_edge("post_board", END)
builder.add_edge("send_email_all", END)

# 체크포인트 기록이 있어야 과거 시점으로 돌아갈 수 있습니다.
delivery_app = builder.compile(checkpointer=InMemorySaver())

# 컴파일한 노드와 연결을 이미지로 확인합니다.
display(Image(delivery_app.get_graph().draw_mermaid_png()))

In [ ]:
# 실행 횟수를 처음 상태로 돌립니다.
for name in DELIVERY_RUNS:
    DELIVERY_RUNS[name] = 0

fork_session = f"l3-fork-{run_stamp}"
fork_config = make_config(fork_session, fork_session, ["day56", "lesson03", "time-travel", "v1-original"])

# 첫 실행: route_channel이 email_all을 골라 전체 메일이 나갑니다.
delivery_result = delivery_app.invoke({"briefing": "AP-500 주간 시장 브리핑 (시연용 본문)", "channel": "", "sent": []}, fork_config)

print("고른 채널:", delivery_result["channel"], "| 보낸 채널:", delivery_result["sent"])
print("노드별 실행 횟수:", DELIVERY_RUNS)

**체크포인트 기록 읽기:** 기록은 최신부터 나옵니다. `metadata`의 `source`는 그 체크포인트를 만든 곳입니다. `input`은 시작 입력, `loop`는 노드 실행, `update`는 `update_state`입니다. 기록을 갈래 전후로 두 번 보므로 출력 함수를 하나 둡니다.

In [ ]:
# [보조 함수] 체크포인트 기록 출력: 최신부터 단계, 만든 곳, 다음 노드, channel, sent를 한 줄씩 출력합니다.
def print_history(app, config):
    """thread의 체크포인트를 최신부터 한 줄씩 출력합니다."""
    for snapshot in app.get_state_history(config):
        # 시작 입력 체크포인트에는 아직 State 값이 없으므로 get으로 읽습니다.
        print("step", snapshot.metadata["step"], "|", snapshot.metadata["source"], "| next:", snapshot.next,
              "| channel:", snapshot.values.get("channel"), "| sent:", snapshot.values.get("sent"))


print_history(delivery_app, fork_config)

In [ ]:
# 잘못된 결정 직후의 체크포인트: 다음 노드가 send_email_all인 지점입니다.
wrong_snapshot = None
for snapshot in delivery_app.get_state_history(fork_config):
    if snapshot.next == ("send_email_all",):
        wrong_snapshot = snapshot
        break

print("다음 노드:", wrong_snapshot.next, "| 그때 결정한 channel:", wrong_snapshot.values["channel"])

In [ ]:
# 결정 값만 고친 새 체크포인트를 만듭니다. route_channel이 쓴 값으로 치므로 pick_channel이 다시 판단합니다.
branch_config = delivery_app.update_state(wrong_snapshot.config, {"channel": "board"}, as_node="route_channel")

branch_snapshot = delivery_app.get_state(branch_config)
print("고친 뒤 다음 노드:", branch_snapshot.next, "| channel:", branch_snapshot.values["channel"])

In [ ]:
# Langfuse 기록용 설정에 갈래 체크포인트의 위치(thread_id, checkpoint_id)만 바꿔 넣습니다.
branch_run_config = make_config(fork_session, fork_session, ["day56", "lesson03", "time-travel", "v2-fork"])
branch_run_config["configurable"] = branch_config["configurable"]

# 입력 None: 갈래 체크포인트의 다음 노드(post_board)부터 실행합니다.
branch_result = delivery_app.invoke(None, branch_run_config)

print("보낸 채널:", branch_result["sent"])
print("노드별 실행 횟수:", DELIVERY_RUNS)
print("trace:", trace_url(branch_run_config))

In [ ]:
# 갈래를 실행한 뒤의 기록 전체: 원래 갈래(email_all)의 체크포인트도 그대로 남아 있습니다.
print_history(delivery_app, fork_config)

**확인할 점:**

- `노드별 실행 횟수`는 `{'route_channel': 1, 'post_board': 1, 'send_email_all': 1}`입니다. 첫 실행 뒤와 비교하면 `post_board`만 0에서 1로 늘었습니다.
- 기록에는 원래 갈래(`step 2 | loop`, `channel: email_all`)와 새 갈래(`step 2 | update`, `step 3 | loop`, `sent: ['board']`)가 함께 있습니다.

결정 노드 `route_channel`은 다시 실행되지 않고, 고친 `channel` 값으로 `post_board`만 실행되었습니다.

`send_email_all`도 1번 그대로입니다. 원래 갈래에서 이미 나간 전체 메일은 되돌려지지 않습니다. fork는 State 기록을 갈라 다시 실행할 뿐, 외부 시스템에 이미 일어난 부작용을 취소하지 않습니다. 또 갈래 지점 뒤의 노드는 모두 다시 실행되므로, 원래 갈래에서 이미 성공한 게시 노드가 갈래 지점 뒤에 있으면 다시 게시합니다. 4-4의 멱등 키는 fork에도 필요합니다.

## 5. 운영 점검표

### 5-1. 실패와 관측 신호, 장치

| 실패 | 관측에서 보이는 신호 (궤적·Langfuse) | 장치 | 절 |
|---|---|---|---|
| 핑퐁 | 같은 두 노드가 번갈아 반복, Agent Graph의 방문 횟수가 큼, 끝에 `GraphRecursionError` | `recursion_limit` 직접 지정, 라운드 가드, 같은 반려 사유 감지, `RemainingSteps` | 교안 02 3-1, 3-2 |
| 에러 없는 중단 | 예외 없이 끝났는데 와야 할 노드가 궤적에 없음, 결과 필드가 비어 있음 | 대기 엣지 대신 `defer=True`, 끝난 뒤 결과 필드 검사 | 교안 02 3-3 |
| 일시 오류 | 같은 노드 기록이 ERROR 뒤 성공 | `RetryPolicy(retry_on=일시 오류)` | 교안 02 4 |
| 영구 오류 | 재시도 없이 ERROR 한 번 | 재시도하지 않음, 실패 기록이나 폴백 | 2, 3 (교안 02 4-3) |
| 응답 없음 | 노드 하나의 지연이 `timeout` 값에서 끊기고 `NodeTimeoutError` | async 노드 `timeout` + `RetryPolicy`, 재시도 예산 | 교안 02 5 |
| 겹친 재시도 | generation 하나의 지연이 길고, 같은 노드의 재시도 기록마다 그런 generation이 반복됨. 모델 서버 요청 수가 trace의 generation 수보다 많음 | 재시도는 한 계층에 (`ChatOpenAI(max_retries=0)` 또는 노드 `retry_on`에서 모델 오류 제외) | 교안 02 5-3 |
| 병렬 부분 실패 | 병렬 분기 하나가 ERROR, 다음 노드 기록 없음 | 노드 안 `try/except` + `failures` 리듀서, 결과에 실패를 밝힘 | 2 |
| 필수 자료 실패 뒤 계속 진행 | 핵심 자료가 `failures`에 있는데 `synthesizer`·`writer`가 실행됨 | 진행 판단 노드: `degraded`면 누락을 밝히고 진행, `held`면 발행 보류, `failed`면 중단. `outcome`을 State에 남김 | 2-7 |
| 단독 노드 실패 | 재시도 기록 뒤 `__error_handler__<노드>` | `error_handler` + `Command(goto)` | 3 |
| 실행 중단 | 같은 세션에 실패 trace와 이어 하기 trace | 체크포인터 + 같은 `thread_id`로 `invoke(None, config)` | 4-1~4-3 |
| 중복 부작용 | 게시 호출 기록 두 번, 게시판 글 두 개 | 멱등 키 | 4-4 |
| 잘못된 결정 | 결정 노드의 출력(고른 채널·워커)이 틀리고 그 뒤 노드가 잘못된 일을 함 | 결정 직후 체크포인트에서 `update_state(..., as_node=결정 노드)` 뒤 `invoke(None, 새 config)` | 4-5 |
| 같은 외부 API의 연속 실패 | 같은 API 노드의 ERROR가 짧은 간격으로 이어지고, 재시도마다 지연과 호출 수가 늘어남 | 서킷 브레이커(연속 실패 N번이면 한동안 호출하지 않고 바로 폴백). 내장 기능이 없어 State나 전역 카운터로 직접 만듦 | - |

### 5-2. 사람에게 넘길 조건

자동 복구로 해결되지 않거나 되돌릴 수 없는 일 직전이면 사람이 판단하게 멈춥니다. 51일차의 `interrupt()`로 멈추고 사람의 결정을 받아 이어 갈 수 있습니다.

| 조건 | 브리핑 팀 예 |
|---|---|
| 라운드 상한 도달 | writer와 reviewer가 상한만큼 주고받아도 통과 못 함 |
| 같은 반려 사유 반복 (핑퐁 감지) | "300자를 넘습니다"가 두 번째로 나옴 |
| 필수 자료 실패 (`held`) | 이번 주 경쟁사 가격 조사가 `failures`에 있음 (2-7) |
| 폴백 값을 쓴 채로 게시 직전 | 지난주 가격으로 만든 보고를 게시하려 함 |
| 되돌릴 수 없는 행동 직전 | 외부 고객에게 메일 발송, 결제 |

## 6. 핵심 정리

- **병렬 부분 실패:** `Send` 작업 하나의 예외는 그 단계 전체를 실패시킵니다. 노드 안에서 영구 오류를 `failures`에 기록하고, 일시 오류는 다시 올려 재시도하게 하며, 브리핑에 수집하지 못한 자료를 밝힙니다.
- **실패 뒤 진행 판단:** 격리한 뒤에는 LLM 없는 판단 노드가 선택 자료만 빠지면 `degraded`(축소 진행), 필수 자료(가격)가 빠지면 `held`(발행 보류), 모두 빠지면 `failed`(중단)로 정하고 `outcome`과 이유를 State에 남깁니다. `held`와 `failed`는 예외 없는 정상 종료라 `outcome`을 봐야 알 수 있습니다.
- **폴백 경로:** `error_handler`는 재시도 소진 뒤 실행되며 `Command(goto=...)`로 다음에 실행할 노드를 정해야 합니다. dict만 반환하면 그래프가 그 자리에서 끝납니다. 이 버전에서는 단독 노드에 씁니다.
- **이어 하기와 멱등성:** 체크포인터와 같은 `thread_id`로 `invoke(None, config)`를 부르면 실패한 노드부터 다시 실행하고 성공한 조사는 다시 하지 않습니다. `get_state().next`와 `tasks[].error`(문자열)로 멈춘 곳을 읽습니다. 병렬 단계에서 취소된 작업은 다시 실행됩니다. 다시 실행될 수 있는 부작용 노드에는 멱등 키(예: `weekly-2026-41`)를 넣습니다.
- **과거 결정에서 갈래 만들기(fork):** `get_state_history`로 잘못된 결정 직후 체크포인트를 찾아 `update_state(snapshot.config, 고칠 값, as_node=결정 노드)`로 값만 고치고, 돌려받은 config로 `invoke(None, config)`를 부릅니다. 결정 노드와 그 앞은 다시 실행되지 않고, 갈래 지점 뒤의 노드는 다시 실행되며, 이미 일어난 부작용은 되돌려지지 않습니다.